# 📚 Notebook 03 — Risk Metrics Deep Dive

**Phase 1: Foundations** · Prerequisites: Notebook 02 (indicators, log returns)

---

## 🎯 Learning Objectives

After this notebook, you will be able to:

1. Compute **Sharpe**, **Sortino**, and **Calmar** ratios from raw returns
2. Understand why our bot uses a **composite score**: Sortino 40% + Sharpe 30% + Calmar 30%
3. Calculate **Maximum Drawdown** and the equity curve
4. Derive **standard errors** for risk ratios using the **Delta method**
5. Evaluate whether a backtest result is statistically significant

In [ ]:
# ── Environment Setup ──
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from datetime import datetime, timezone, timedelta
import requests

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# ── Generate sample returns ──
def fetch_btc_hourly(days: int = 90) -> pd.DataFrame:
    end_ms = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_ms = end_ms - days * 86_400_000
    all_rows = []
    cursor = start_ms
    while cursor < end_ms:
        resp = requests.get("https://api.binance.com/api/v3/klines",
                           params={"symbol": "BTCUSDT", "interval": "1h",
                                   "startTime": cursor, "endTime": end_ms, "limit": 1000},
                           timeout=10)
        resp.raise_for_status()
        rows = resp.json()
        if not rows: break
        all_rows.extend(rows)
        cursor = int(rows[-1][0]) + 3_600_000
        if len(rows) < 1000: break
    df = pd.DataFrame(all_rows, columns=["open_time","open","high","low","close","volume",
                                          "close_time","quote_vol","trades","taker_base","taker_quote","ignore"])
    for col in ["open","high","low","close","volume"]:
        df[col] = df[col].astype(float)
    df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df = df.set_index("timestamp")[["open","high","low","close","volume"]]
    return df

df = fetch_btc_hourly(90)
close = df["close"]
log_returns = np.log(close / close.shift(1)).dropna()
print(f"✅ Loaded {len(log_returns)} hourly log returns")
print(f"   Mean: {log_returns.mean():.6f}  Std: {log_returns.std():.6f}")

---

## 📐 Section 1: Sharpe Ratio

The Sharpe ratio measures **return per unit of total risk**:

$$\text{Sharpe} = \frac{\bar{r} - r_f}{\sigma}$$

where:
- $\bar{r}$ = mean return per period
- $r_f$ = risk-free rate per period (we use 0 for crypto)
- $\sigma$ = standard deviation of returns

### Annualization

For hourly data with $N = 8{,}760$ hours per year:

$$\text{Sharpe}_{\text{ann}} = \text{Sharpe}_{\text{hourly}} \times \sqrt{N} = \frac{\bar{r}}{\sigma} \times \sqrt{8760}$$

### Interpretation

| Sharpe | Quality |
|--------|--------|
| < 0 | Losing money |
| 0–1 | Below average |
| 1–2 | Good |
| 2–3 | Very good |
| > 3 | Exceptional (or overfitted!) |

In [ ]:
# ── Sharpe Ratio from scratch ──

def sharpe_ratio(returns: pd.Series, periods_per_year: int = 8760) -> float:
    """Annualized Sharpe ratio (excess returns / volatility).
    
    Used in the bot's composite scoring:
    Sortino (40%) + Sharpe (30%) + Calmar (30%)
    """
    if returns.std() == 0:
        return 0.0
    return (returns.mean() / returns.std()) * np.sqrt(periods_per_year)

sharpe = sharpe_ratio(log_returns)
print(f"Annualized Sharpe Ratio: {sharpe:.4f}")
print(f"\nBreaking down:")
print(f"  Mean hourly return: {log_returns.mean():.8f}")
print(f"  Hourly volatility:  {log_returns.std():.8f}")
print(f"  Hourly Sharpe:      {log_returns.mean() / log_returns.std():.6f}")
print(f"  × √8760 =          {sharpe:.4f}")

---

## 📐 Section 2: Sortino Ratio — Penalizing Downside Only

The Sharpe ratio penalizes **all** volatility equally — but upside volatility is good! The Sortino ratio only penalizes **downside deviation**:

$$\text{Sortino} = \frac{\bar{r} - r_f}{\sigma_d}$$

where the **downside deviation** is:

$$\sigma_d = \sqrt{\frac{1}{n} \sum_{t=1}^{n} \min(r_t - r_f, 0)^2}$$

### Why Sortino Gets 40% Weight

Our bot uses Sortino as the **primary** metric (40% weight) because:
- Crypto has highly asymmetric returns (occasional large pumps)
- We don't want to penalize large upside moves
- Downside risk is what actually blows up accounts

In [ ]:
# ── Sortino Ratio from scratch ──

def sortino_ratio(returns: pd.Series, periods_per_year: int = 8760) -> float:
    """Annualized Sortino ratio (excess return / downside deviation).
    
    This is the HIGHEST-WEIGHTED metric in our composite score (40%).
    """
    # Downside deviation: std of negative returns only
    downside = returns.clip(upper=0)  # Keep only negative returns
    downside_dev = np.sqrt((downside ** 2).mean())  # RMS of downside
    
    if downside_dev == 0:
        return 0.0
    
    return (returns.mean() / downside_dev) * np.sqrt(periods_per_year)

sortino = sortino_ratio(log_returns)
print(f"Annualized Sortino Ratio: {sortino:.4f}")
print(f"Annualized Sharpe Ratio:  {sharpe:.4f}")
print(f"\nSortino / Sharpe = {sortino / sharpe:.2f}x" if sharpe != 0 else "")
print(f"(Sortino > Sharpe when positive returns are more volatile than negative)")

---

## 📐 Section 3: Maximum Drawdown & Calmar Ratio

### Maximum Drawdown (MaxDD)

The **maximum drawdown** is the largest peak-to-trough decline in the cumulative return:

$$\text{MaxDD} = \max_{t} \left( \frac{\text{Peak}_t - \text{Value}_t}{\text{Peak}_t} \right)$$

### Calmar Ratio

$$\text{Calmar} = \frac{\text{Annualized Return}}{|\text{MaxDD}|}$$

Calmar answers: *"For every 1% of maximum drawdown pain, how much annual return do I get?"*

It gets **30% weight** in our composite because it captures **tail risk** that Sharpe and Sortino miss.

In [ ]:
# ── Maximum Drawdown and Calmar Ratio ──

def compute_drawdown(returns: pd.Series) -> pd.DataFrame:
    """Compute the drawdown series from returns."""
    # Cumulative wealth (equity curve)
    cumulative = (1 + returns).cumprod()
    
    # Running peak
    peak = cumulative.cummax()
    
    # Drawdown: how far below the peak
    drawdown = (cumulative - peak) / peak
    
    return pd.DataFrame({
        'cumulative': cumulative,
        'peak': peak,
        'drawdown': drawdown,
    })

def max_drawdown(returns: pd.Series) -> float:
    """Maximum drawdown (as a positive fraction, e.g., 0.15 = 15% drawdown)."""
    dd_df = compute_drawdown(returns)
    return abs(dd_df['drawdown'].min())

def calmar_ratio(returns: pd.Series, periods_per_year: int = 8760) -> float:
    """Annualized Calmar ratio (annual return / max drawdown).
    
    Gets 30% weight in our composite score.
    """
    ann_return = returns.mean() * periods_per_year
    mdd = max_drawdown(returns)
    if mdd == 0:
        return 0.0
    return ann_return / mdd

mdd = max_drawdown(log_returns)
calmar = calmar_ratio(log_returns)

print(f"Maximum Drawdown: {mdd:.2%}")
print(f"Calmar Ratio:     {calmar:.4f}")

In [ ]:
# ── Equity curve and drawdown visualization ──
dd_df = compute_drawdown(log_returns)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[2, 1], sharex=True)

# Equity curve
ax1.plot(dd_df['cumulative'].index, dd_df['cumulative'], label='Cumulative Return', color='#2196F3', linewidth=1.5)
ax1.plot(dd_df['peak'].index, dd_df['peak'], label='Peak', color='gray', linewidth=1, linestyle='--', alpha=0.5)
ax1.set_ylabel('Equity (starting at 1.0)')
ax1.set_title('BTC/USDT — Equity Curve and Drawdown', fontsize=14)
ax1.legend()

# Drawdown
ax2.fill_between(dd_df['drawdown'].index, 0, dd_df['drawdown'], color='red', alpha=0.3)
ax2.plot(dd_df['drawdown'].index, dd_df['drawdown'], color='darkred', linewidth=1)
ax2.set_ylabel('Drawdown')
ax2.set_title(f'Drawdown (Max: {mdd:.2%})', fontsize=11)

# Mark the maximum drawdown point
worst_idx = dd_df['drawdown'].idxmin()
ax2.annotate(f'Max DD: {mdd:.2%}', xy=(worst_idx, dd_df['drawdown'].min()),
             fontsize=11, color='darkred', fontweight='bold',
             xytext=(10, -30), textcoords='offset points',
             arrowprops=dict(arrowstyle='->', color='darkred'))

plt.tight_layout()
plt.show()

---

## 📐 Section 4: Composite Score

Our bot evaluates strategy performance using a **weighted composite**:

$$\text{Score} = 0.40 \times \text{Sortino} + 0.30 \times \text{Sharpe} + 0.30 \times \text{Calmar}$$

### Why These Weights?

| Metric | Weight | Captures | Weakness |
|--------|--------|----------|----------|
| Sortino | 40% | Downside risk | Ignores correlation structure |
| Sharpe | 30% | Total risk-adjusted return | Penalizes upside vol |
| Calmar | 30% | Tail/drawdown risk | Single-point estimate |

In [ ]:
# ── Composite Score ──

def composite_score(returns: pd.Series, periods_per_year: int = 8760) -> dict:
    """Compute the bot's composite performance score.
    
    Weights: Sortino 40% + Sharpe 30% + Calmar 30%
    Reference: bot/backtest/core_module_backtester.py
    """
    s = sharpe_ratio(returns, periods_per_year)
    so = sortino_ratio(returns, periods_per_year)
    c = calmar_ratio(returns, periods_per_year)
    composite = 0.40 * so + 0.30 * s + 0.30 * c
    
    return {
        'sharpe': s,
        'sortino': so,
        'calmar': c,
        'composite': composite,
        'max_drawdown': max_drawdown(returns),
        'annual_return': returns.mean() * periods_per_year,
    }

scores = composite_score(log_returns)
print("═" * 50)
print("       COMPOSITE PERFORMANCE SCORECARD")
print("═" * 50)
print(f"  Sharpe  (30%):  {scores['sharpe']:>8.4f}")
print(f"  Sortino (40%):  {scores['sortino']:>8.4f}")
print(f"  Calmar  (30%):  {scores['calmar']:>8.4f}")
print(f"  ─────────────────────────────")
print(f"  COMPOSITE:      {scores['composite']:>8.4f}")
print(f"")
print(f"  Ann. Return:    {scores['annual_return']:>8.2%}")
print(f"  Max Drawdown:   {scores['max_drawdown']:>8.2%}")
print("═" * 50)

---

## 📐 Section 5: Delta Method for Standard Errors

A Sharpe ratio of 1.5 means nothing if the **uncertainty** is ±2.0. We need **standard errors** to assess statistical significance.

### The Delta Method

For a function $g(\mu, \sigma)$ of sample statistics, the Delta method approximates the variance:

$$\text{Var}[g(\hat{\mu}, \hat{\sigma})] \approx \nabla g^T \cdot \Sigma \cdot \nabla g$$

### Sharpe Ratio Standard Error

For $\text{SR} = \mu / \sigma$, the SE under normality is approximately:

$$\text{SE}(\text{SR}) \approx \sqrt{\frac{1 + \frac{\text{SR}^2}{2}}{n}}$$

This accounts for estimation error in both $\mu$ and $\sigma$.

### Sortino Ratio Standard Error

The Sortino SE is more complex because downside deviation uses only negative returns:

$$\text{SE}(\text{Sortino}) \approx \frac{1}{\sigma_d} \sqrt{\frac{\sigma^2}{n} + \frac{\mu^2 \cdot \text{Var}(\sigma_d)}{\sigma_d^2}}$$

We derive this using matrix calculus in the strategy notes.

In [ ]:
# ── Delta Method Standard Errors ──

def sharpe_standard_error(returns: pd.Series, periods_per_year: int = 8760) -> dict:
    """Compute Sharpe ratio with its standard error using the Delta method.
    
    Reference: strategy_notes_ZH.md Section 5 (Delta Method derivation)
    """
    n = len(returns)
    mu = returns.mean()
    sigma = returns.std()
    
    if sigma == 0:
        return {'sharpe': 0.0, 'se': 0.0, 't_stat': 0.0, 'p_value': 1.0}
    
    # Per-period Sharpe
    sr = mu / sigma
    
    # Delta method SE: sqrt((1 + SR²/2) / n)
    se_per_period = np.sqrt((1 + sr**2 / 2) / n)
    
    # Annualize both
    sr_ann = sr * np.sqrt(periods_per_year)
    se_ann = se_per_period * np.sqrt(periods_per_year)
    
    # t-statistic: is the Sharpe significantly different from 0?
    t_stat = sr / se_per_period
    p_value = 2 * (1 - stats.norm.cdf(abs(t_stat)))
    
    return {
        'sharpe': sr_ann,
        'se': se_ann,
        'ci_lower': sr_ann - 1.96 * se_ann,
        'ci_upper': sr_ann + 1.96 * se_ann,
        't_stat': t_stat,
        'p_value': p_value,
    }

result = sharpe_standard_error(log_returns)
print("Sharpe Ratio with Confidence Interval:")
print(f"  Sharpe:        {result['sharpe']:.4f}")
print(f"  Std Error:     {result['se']:.4f}")
print(f"  95% CI:        [{result['ci_lower']:.4f}, {result['ci_upper']:.4f}]")
print(f"  t-statistic:   {result['t_stat']:.4f}")
print(f"  p-value:       {result['p_value']:.4f}")
print(f"\n  Significant at 5%? {'YES ✅' if result['p_value'] < 0.05 else 'NO ❌'}")

In [ ]:
# ── Visualize: Sharpe vs sample size ──
# How many data points do you need for a reliable estimate?

sample_sizes = [100, 200, 500, 720, 1000, 1440, 2160, len(log_returns)]
results = []
for n in sample_sizes:
    if n > len(log_returns):
        continue
    subset = log_returns.iloc[-n:]
    r = sharpe_standard_error(subset)
    results.append({'n': n, 'days': n/24, **r})

res_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(12, 5))
ax.errorbar(res_df['days'], res_df['sharpe'], yerr=1.96*res_df['se'],
            fmt='o-', capsize=5, capthick=2, markersize=8, color='#2196F3')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Sample Size (days)')
ax.set_ylabel('Annualized Sharpe Ratio')
ax.set_title('Sharpe Ratio Estimate vs Sample Size (with 95% CI)', fontsize=14)
for _, row in res_df.iterrows():
    ax.annotate(f"n={int(row['n'])}", (row['days'], row['sharpe']),
                textcoords='offset points', xytext=(0, 12), fontsize=8, ha='center')
plt.tight_layout()
plt.show()

print("\n📌 Rule of thumb: You need ~2000+ hourly samples for a reasonably tight CI")

---

## 📐 Section 6: Rolling Metrics — Regime Sensitivity

Static metrics hide **time-varying performance**. Rolling metrics reveal how the strategy behaves across different market regimes.

In [ ]:
# ── Rolling Sharpe and Sortino ──

def rolling_sharpe(returns: pd.Series, window: int = 168, periods_per_year: int = 8760) -> pd.Series:
    """Rolling annualized Sharpe ratio.
    Window=168 hours = 7 days.
    """
    rolling_mean = returns.rolling(window).mean()
    rolling_std = returns.rolling(window).std()
    return (rolling_mean / rolling_std) * np.sqrt(periods_per_year)

def rolling_sortino(returns: pd.Series, window: int = 168, periods_per_year: int = 8760) -> pd.Series:
    """Rolling annualized Sortino ratio."""
    rolling_mean = returns.rolling(window).mean()
    downside = returns.clip(upper=0)
    rolling_dd = np.sqrt((downside ** 2).rolling(window).mean())
    return (rolling_mean / rolling_dd) * np.sqrt(periods_per_year)

roll_sharpe = rolling_sharpe(log_returns, 168)  # 7-day window
roll_sortino = rolling_sortino(log_returns, 168)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[1, 1], sharex=True)

ax1.plot(close.index, close, color='#333', linewidth=1)
ax1.set_ylabel('Price (USDT)')
ax1.set_title('BTC/USDT Price and Rolling Risk Metrics (7-day window)', fontsize=14)

ax2.plot(roll_sharpe.index, roll_sharpe, label='Rolling Sharpe', color='#2196F3', linewidth=1)
ax2.plot(roll_sortino.index, roll_sortino, label='Rolling Sortino', color='#4CAF50', linewidth=1)
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.fill_between(roll_sharpe.index, 0, roll_sharpe,
                 where=roll_sharpe > 0, alpha=0.1, color='green')
ax2.fill_between(roll_sharpe.index, 0, roll_sharpe,
                 where=roll_sharpe < 0, alpha=0.1, color='red')
ax2.set_ylabel('Annualized Ratio')
ax2.legend()
ax2.set_ylim(-10, 10)  # Clip extreme values for readability

plt.tight_layout()
plt.show()

---

## 📐 Section 7: The Circuit Breaker Connection

The bot's **circuit breaker** (`bot/risk/circuit_breaker.py`) uses drawdown as a real-time risk control:

| Level | Drawdown | Action |
|-------|---------|--------|
| L1 | ≥ 3% | Reduce position sizes |
| L2 | ≥ 5% | **HALT all trading** |

This is why MaxDD and Calmar matter — they measure exactly the risk that triggers emergency stops.

In [ ]:
# ── Circuit Breaker simulation ──
from bot.risk.circuit_breaker import CircuitBreaker

cb = CircuitBreaker(l1_threshold=0.03, l2_threshold=0.05)

# Simulate the circuit breaker on historical drawdowns
dd_series = compute_drawdown(log_returns)['drawdown']
statuses = [cb.check(abs(dd)) for dd in dd_series]

status_counts = pd.Series(statuses).value_counts()
print("Circuit Breaker Status Distribution:")
for status, count in status_counts.items():
    pct = count / len(statuses) * 100
    emoji = {'ok': '✅', 'reduce': '⚠️', 'halt': '🛑'}.get(status, '')
    print(f"  {emoji} {status:>6}: {count:>5} periods ({pct:.1f}%)")

---

## 🔬 Exercises

### Exercise 1: Multi-Asset Scorecard 🔬

Fetch 90-day hourly data for BTC, ETH, and SOL. Compute the composite score for each. Which asset would the bot rank highest?

In [ ]:
# ── Exercise 1: Your code here ──

# YOUR CODE HERE

### Exercise 2: Minimum Track Record Length ⭐

Using the Delta method, calculate the minimum number of hourly observations needed for a Sharpe ratio of 1.0 to be significant at the 5% level. How many days is that?

In [ ]:
# ── Exercise 2: Your code here ──

# Hint: Solve for n in t_stat = SR / SE(SR) > 1.96
# YOUR CODE HERE

---

## ✅ Knowledge Check

1. What is the key difference between Sharpe and Sortino?
2. Why does our bot weight Sortino at 40%?
3. A backtest shows Sharpe = 2.5 with SE = 3.0. Is this a good strategy?
4. What is the circuit breaker L2 threshold?
5. How do you annualize an hourly Sharpe ratio?

<details>
<summary>Click for answers</summary>

1. Sharpe penalizes ALL volatility; Sortino only penalizes DOWNSIDE volatility
2. Crypto has asymmetric returns — large upside moves shouldn't be penalized, and downside risk is what blows up accounts
3. No — the 95% CI is [2.5 - 5.88, 2.5 + 5.88] = [-3.38, 8.38], which includes 0. The result is not statistically significant
4. 5% drawdown → halt all trading
5. Multiply by √8760 (hours per year)

</details>

---

## 🔗 Next: Notebook 04 — Momentum Strategy

With data, indicators, and risk metrics in place, you're ready to build your first **trading strategy**. In Notebook 04, we'll implement the momentum signal engine — ranking assets by recent performance and filtering with RSI/EMA/volume.

**Open:** `04_Momentum_Strategy.ipynb`